# Hybrid Search

This notebook combines **dense vector search** (FAISS embeddings) with **sparse keyword search** (BM25) using an ensemble retriever. Dense search can find semantically related wording; BM25 is strong for exact terms, names, and identifiers. Combining them can improve retrieval when either method alone misses relevant documents. The retrieved documents are then passed to a RAG chain to generate an answer.

In [9]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

In [10]:
text="""
LangChain is a framework used to build applications powered by Large Language Models (LLMs). It provides document loaders, text splitters, retrievers, and prompt templates for creating RAG applications.
LangChain supports integration with different LLM providers and vector databases. Developers can use it to retrieve relevant documents and provide contextual information to language models.
Python is a popular, high-level programming language known for its simple and readable syntax. It is widely used in web development, automation, data analysis, and artificial intelligence.
Python supports object-oriented, procedural, and functional programming paradigms. Popular libraries such as Pandas, NumPy, and Scikit-learn help developers process data and build machine learning models.
The Taj Mahal is a famous historical monument located in Agra, India. Built by Mughal emperor Shah Jahan in memory of his wife Mumtaz Mahal, it is renowned for its white marble architecture.
Photosynthesis is the process by which green plants use sunlight, water, and carbon dioxide to produce glucose and release oxygen. This process is essential for sustaining life and maintaining oxygen levels in Earth's atmosphere.
"""

In [11]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
docs = [
    Document(page_content= doc.strip()) for doc in text.splitlines() if doc.strip()
]

retrivier = FAISS.from_documents(docs, embedding=embeddings)
dence_retriver = retrivier.as_retriever()

dence_retriver

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x11dff3d90>, search_kwargs={})

In [15]:
sparse_retriver = BM25Retriever.from_documents(docs)
sparse_retriver.k = 3

In [16]:
from langchain_classic.retrievers import EnsembleRetriever
compbine_retriever = EnsembleRetriever(
    retrievers=[dence_retriver, sparse_retriver],
    weights=[0.5, 0.5]
)


In [20]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

In [25]:
prompt = ChatPromptTemplate.from_template(
    template=""" Please provide answer of question based on provided context
    Context: {context}
    Question: {input}
    """
)

llm = ChatOpenAI(
    model="gpt-4o-mini"
)

chain_document = create_stuff_documents_chain(llm, prompt)
conv_retrival = create_retrieval_chain(retriever=compbine_retriever, combine_docs_chain=chain_document)

In [30]:
query = "Can you help to understand Python with langchain ?"
result = conv_retrival.invoke({"input": query})
result["answer"]
# for i, doc in enumerate(result):
#     print(f"Doc {i+1}: {doc.page_content}")

"Absolutely! Python is a versatile programming language that can be effectively used with LangChain to build applications that leverage Large Language Models (LLMs). Here’s a breakdown of how Python interacts with LangChain:\n\n1. **Framework Overview**: LangChain is a framework specifically designed for building applications that utilize LLMs. It includes various components such as document loaders, text splitters, retrievers, and prompt templates that facilitate the Retrieval-Augmented Generation (RAG) approach.\n\n2. **Object-Oriented and Libraries**: Python's support for object-oriented programming makes it easier to design complex applications. In LangChain, you can create classes to encapsulate different functionalities like loading documents and processing them for LLMs. Additionally, libraries like Pandas and NumPy can be used for data manipulation, which complements the capabilities of LangChain when preparing data for LLMs.\n\n3. **Integration with Data Processing**: If you n